In [17]:
import math
import numpy
import random

In [18]:
# activation functions
def sigmoid(x):
    value = 1 / (1+math.e**-x)
    return value
def tanh(x):
    value = (math.e**x - math.e**-x)/(math.e**x + math.e**-x)
    return value
def ReLU(x):
    return (max(x,0))
def lRelu(x):
    return(max(x*0.1,x))
def pRelu(x,a):
    return(max(a*x,x))

# respective derivatives
def sigmoid_derivative(output):
    value = output * (1-output)
    return value
def sigmoid_derivative(output):
    value = output * (1-output)
    return value
def tanh_derivative(output):
    value = 1.0 - (output ** 2)
    return value
def Relu_derivative(output):
    value = 1 if output > 0 else 0
    return value
def lRelu_derivative(output):
    value = 1 if output>0 else 0.1
    return value
def pRelu_derivative(output, a):
    value = 1 if output > 0 else a
    return value

ACTIVATIONS = {
    'sigmoid': (sigmoid, sigmoid_derivative),
    'relu': (ReLU, Relu_derivative),
    'tanh': (tanh, tanh_derivative)
}

In [19]:
class neuron:
    def __init__(self, num_inputs, activation = "sigmoid"):
        self.weights = [random.uniform(-1,1) for _ in range(num_inputs)]
        self.bias = 0
        self.inputs = []
        self.total = 0
        self.output = 0
        self.error = 0
        self.activation, self.derivative = ACTIVATIONS[activation]

    def forward(self, inputs):
        self.inputs = inputs
        self.total = sum(w * x for w, x in zip(self.weights, inputs)) + self.bias
        self.output =  sigmoid(self.total)
        return self.output
        
def mse(output, actual):
    value = (actual-output)**2
    return value


In [20]:
class layer:
    def __init__(self, num_neurons, num_inputs_per_neuron, activation="sigmoid"):
        self.neurons = [neuron(num_inputs_per_neuron, activation=activation) for _ in range(num_neurons)]
    def forward(self, inputs):
        return[neuron.forward(inputs) for neuron in self.neurons]

class nn:
    def __init__(self, num_inputs, hidden_layer_num, hidden_layer_size, num_outputs):
        # self.first_layer = layer(hidden_layer_size,num_inputs)
        # self.hidden_layer = layer(hidden_layer_size,hidden_layer_size)
        # self.final_layer = layer(num_outputs, hidden_layer_size)
        self.layers = []
        self.layers.append(layer(hidden_layer_size,num_inputs))
        for _ in range(hidden_layer_num-1):
            self.layers.append(layer(hidden_layer_size,hidden_layer_size))
        self.layers.append(layer(num_outputs, hidden_layer_size))
    
    def forward(self,inputs):
        x = inputs
        for layer in self.layers:
            x = layer.forward(x)
        return x
    
    def backward(self,targets):
        output_layer = self.layers[-1]
        for i, neuron in enumerate(output_layer.neurons):
            error = neuron.output - targets[i]
            neuron.error = error*neuron.derivative(neuron.output)

        for l in range(len(self.layers) - 2, -1, -1):
            current_layer = self.layers[l]
            next_layer = self.layers[l+1]
            for i, neuron in enumerate(current_layer.neurons):
                error = sum(next_neuron.weights[i] * next_neuron.error for next_neuron in next_layer.neurons)
                neuron.error  = error * neuron.derivative(neuron.output)
    def update_weights(self, learning_rate=0.5):
        for layer in self.layers:
            for neuron in layer.neurons:
                for j in range(len(neuron.weights)):
                    neuron.weights[j] -= learning_rate * neuron.error * neuron.inputs[j]
                neuron.bias -= learning_rate * neuron.error


In [21]:
# --- TEST IT ON THE XOR PROBLEM ---

# Inputs and corresponding XOR targets
X = [[0, 0], [0, 1], [1, 0], [1, 1]]
Y = [[0],    [1],    [1],    [0]]

# 2 inputs, 1 hidden layer with 3 neurons, 1 output neuron
model = nn(num_inputs=2, hidden_layer_num=1, hidden_layer_size=3, num_outputs=1)

# Training loop
epochs = 20000
learning_rate = 0.5

print("Training neural network...")
for epoch in range(epochs):
    total_loss = 0
    for x, y in zip(X, Y):
        output = model.forward(x)
        total_loss += mse(output[0], y[0])
        model.backward(y)
        model.update_weights(learning_rate)
    
    if epoch % 5000 == 0:
        print(f"Epoch {epoch} - Total Loss: {total_loss:.5f}")

print("\nFinal Predictions after training:")
for x, y in zip(X, Y):
    prediction = model.forward(x)[0]
    print(f"Input: {x} -> Predicted: {prediction:.4f} (Target: {y[0]})")

Training neural network...
Epoch 0 - Total Loss: 1.10464
Epoch 5000 - Total Loss: 0.00272
Epoch 10000 - Total Loss: 0.00116
Epoch 15000 - Total Loss: 0.00073

Final Predictions after training:
Input: [0, 0] -> Predicted: 0.0109 (Target: 0)
Input: [0, 1] -> Predicted: 0.9890 (Target: 1)
Input: [1, 0] -> Predicted: 0.9887 (Target: 1)
Input: [1, 1] -> Predicted: 0.0127 (Target: 0)
